# Tutorial 14 — **Pandas vectorizado** y alternativas: **Dask** y **Polars**

**Objetivo**: Trabajar **por columnas** con operaciones **vectorizadas** en Pandas para ganar rendimiento y claridad; y conocer **Dask** (para escalar más allá de la RAM) y **Polars** (alto desempeño en una sola máquina).  
**Qué lograrás**:
- Identificar cuándo *no* conviene `iterrows()`/`apply(axis=1)`.
- Reescribir transformaciones a una versión **vectorizada**.
- Ejecutar un *mini* flujo con **Dask DataFrame**.
- Replicar el mismo flujo con **Polars** en modo *eager* y *lazy*.


<a id="material"></a>
## 1) Material de estudio

<a id="ineficiente"></a>
### 1.1 ¿Por qué iterar filas suele ser ineficiente?

Recorrer un `DataFrame` fila por fila con `for`, `.iterrows()` o `apply(axis=1)` ejecuta el **bucle en Python**, que no está optimizado para millones de iteraciones; en consecuencia, resulta **lento y poco escalable** con datasets medianos o grandes.  
La clave es **trabajar por columnas completas** y delegar el cálculo a librerías optimizadas. 

<a id="vectorizado"></a>
### 1.2 Procesamiento vectorizado en Pandas

Una operación vectorizada **aplica una instrucción sobre una columna/array completo**. Pandas se apoya en **NumPy**, que a su vez ejecuta rutinas en **C** (y BLAS) con memoria contigua y, en muchos casos, **SIMD/paralelización interna**. Por eso, sumar/transformar columnas suele ser **decenas de veces más rápido** que iterar fila a fila.  
**Evitar**: `iterrows()`, `apply(axis=1)` y bucles explícitos sobre filas.  
**Preferir**: operaciones aritméticas/lógicas entre columnas, `mean/sum/max/min`, `map/agg/where/mask`, y funciones de `numpy` (`np.log`, `np.where`, `np.sqrt`, …).

<a id="alternativas"></a>
### 1.3 Alternativas cuando Pandas se queda corto: **Dask** y **Polars**

- **Pandas** carga todo en **RAM**; con archivos de varios GB, el rendimiento cae o no cabe en memoria.  
- **Dask** divide los datos en **particiones** y ejecuta **en paralelo**, permitiendo trabajar con datasets que exceden la RAM, manteniendo una API similar a Pandas.  
- **Polars** (en **Rust**) usa un motor **columnar** y paraleliza por defecto: *muy* rápido para análisis en una sola máquina, con sintaxis expresiva y modo **lazy** (diferido) para optimización de consultas.

<a id="cuando"></a>
### 1.4 ¿Cuándo usar cada herramienta? (regla práctica)

- **Pandas + vectorizado** → datasets **pequeños/medianos** que caben en memoria, exploración y prototipos rápidos.  
- **Dask** → datasets **más grandes que RAM** o flujos que se benefician de **paralelismo** sin abandonar la API estilo Pandas.  
- **Polars** → **máximo rendimiento** en una sola máquina (columnar + paralelismo nativo) y consultas **lazy** optimizadas.

<a id="ejemplos"></a>
## 2) Ejemplos

> **Nota de rendimiento**: ejecuta las celdas en orden. Para este ejemplo usaremos N = 500000 pero pueden variarlo.

In [17]:
# Preparación
import numpy as np, pandas as pd
rng = np.random.default_rng(0)
N = 500000
df = pd.DataFrame({
    "a": rng.random(N),
    "b": rng.random(N)
})
df.head()

,a,b
0,0.636962,0.775295
1,0.269787,0.737937
2,0.040974,0.275778
3,0.016528,0.394414
4,0.813270,0.890251


> **NOTA:** %%timeit -n 1 -r 3 --> sirve para medir el tiempo de ejecución de un fragmento de código, básicamente el código se ejecuta tres veces y cada vez se ejecuta una sola iteración.  

### 2.1 — Fila a fila (iterrows)

In [18]:
%%timeit -n 1 -r 3
# 2.1 — Fila a fila (iterrows) — *muy* lento
out = []
for _, row in df.iterrows():
    out.append(row["a"]*row["b"] + row["a"]**2)
s_iter = pd.Series(out)

18 s ± 136 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [19]:
out = []
for _, row in df.iterrows():
    out.append(row["a"]*row["b"] + row["a"]**2)
s_iter = pd.Series(out)

### 2.2 — apply(axis=1)

In [20]:
%%timeit -n 1 -r 3
# 2.2 — apply(axis=1) — sigue siendo fila a fila
s_apply = df.apply(lambda r: r["a"]*r["b"] + r["a"]**2, axis=1)

4.87 s ± 3.18 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [21]:
s_apply = df.apply(lambda r: r["a"]*r["b"] + r["a"]**2, axis=1)

### 2.3 — Vectorizado puro (NumPy/Pandas)

In [22]:
%%timeit -n 1 -r 3
# 2.3 — Vectorizado puro (NumPy/Pandas)
a = df["a"].to_numpy()
b = df["b"].to_numpy()
s_vec = a*b + a*a  # equivalente a: df["a"]*df["a"] + df["a"]*df["b"]

2.35 ms ± 1.22 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [23]:
s_vec = df["a"]*df["a"] + df["a"]*df["b"]

In [24]:
# Verificación de equivalencia
pd.testing.assert_series_equal(pd.Series(s_vec).reset_index(drop=True),
                               s_apply.reset_index(drop=True),
                               check_names=False)
print("¡Vectorizado y fila-a-fila coinciden!")

¡Vectorizado y fila-a-fila coinciden!


### 2.4 — Dask DataFrame

In [25]:
#!pip install dask[complete]

In [26]:
import dask.dataframe as dd

# Particionar desde un DataFrame existente
ddf = dd.from_pandas(df, npartitions=8)

# Mismo cálculo y una agregación
ddf2 = ddf.assign(c=ddf["a"]*ddf["b"] + ddf["a"]**2)
res = ddf2["c"].mean().compute()
res

0.5835479081290561

### 2.5 — Polars

In [27]:
#!pip install polars

In [28]:
import polars as pl

pl_df = pl.from_pandas(df)

# Modo eager
pl_df2 = pl_df.with_columns(
    c = pl.col("a")*pl.col("b") + pl.col("a")**2
)
pl_df2.head()

a,b,c
f64,f64,f64
0.636962,0.775295,0.899553
0.269787,0.737937,0.271871
0.040974,0.275778,0.012978
0.016528,0.394414,0.006792
0.81327,0.890251,1.385423


In [29]:
# Modo lazy (consulta diferida y optimizada)
q = (pl_df.lazy()
     .with_columns(c = pl.col("a")*pl.col("b") + pl.col("a")**2)
     .select((pl.col("c")).mean().alias("c_mean"))
    )
q.collect()

c_mean
f64
0.583548


<a id="comparacion"></a>
### 2.6 Mini‑guía de comparaciones

| Tarea | Pandas (vectorizado) | Dask | Polars |
|---|---|---|---|
| Datos caben en RAM | SI | SI | SI |
| Datos mayores que RAM | NO | SI | (posible vía *scan*/memmap en algunos casos) |
| API similar a Pandas | SI | SI | NO (sintaxis propia) |
| Paralelismo nativo | (NumPy interno) | SI | SI |
| Lazy/optimizador de consultas | NO | (graphs) | SI |

>  *Nota*: detalles y límites dependen de la configuración y el tamaño real de los datos.

<a id="ideas"></a>
## 3) Ideas finales

- El cuello de botella típico no es **Pandas**, sino *cómo lo usamos*: evitar `iterrows()/apply(axis=1)` y **vectorizar**.  
- Cuando el volumen crece, **Pandas = RAM**. Usa **Dask** para particionar y **Polars** para sacar máximo provecho del hardware local (motor columnar, paralelismo, *lazy*).  
- Dominar estas herramientas te prepara para flujos **eficientes, escalables y modernos**.

<a id="ejercicios"></a>
## 4) Ejercicios

> **Formato**: escribe `print(...)` descriptivos, y comenta brevemente tus conclusiones.

### Ejercicio 1 — Reescritura vectorizada
Convierte una transformación escrita con `apply(axis=1)` a una versión **vectorizada** pura.  
*Pista*: utiliza operaciones entre columnas y funciones de `numpy`.

### Ejercicio 2 — Mini‑pipeline con Dask
Crea un `dask.dataframe` desde un `df` y calcula:  
1. una nueva columna derivada, 2. un `groupby` simple, 3. una agregación.  
Reporta el **`.npartitions`** y discute cómo afecta el rendimiento.





### Ejercicio 3 — Polars en modo lazy
Replica el pipeline anterior en **Polars** usando `.lazy()` → `.collect()`.  
Añade al menos una **proyección** (`select`) y una **agregación**.